{{ badge }}


# Deep learning method for portfolio weights prediction.


## Imports and configuration


In [1]:
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.stats import norm
from sklearn.linear_model import Ridge
from sklearn.preprocessing import MinMaxScaler
import gdown

seed = 42
np.random.seed(seed)
torch.manual_seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Data loading


In [3]:
url = "https://drive.google.com/uc?id=1qngF_Dp2I3bT4grNz5oErCOtPYut7Dj7"
file_path = "Dataset3_PortfolioReplicaStrategy.xlsx"
gdown.download(url, file_path, quiet=False)
data = pd.read_excel(file_path, header=0)

Downloading...
From: https://drive.google.com/uc?id=1qngF_Dp2I3bT4grNz5oErCOtPYut7Dj7
To: /Users/pengrao/Workspace/Fintech/BC3/Dataset3_PortfolioReplicaStrategy.xlsx
100%|██████████| 93.9k/93.9k [00:00<00:00, 2.36MB/s]


## Data preparation

Build the 11-futures return matrix `X` and the Monster Index target `y`.


In [ ]:
# Normalize the loaded frame: first column is the date, the rest are bare tickers
# (4 indices + 11 futures). Add the conventional Bloomberg-style suffixes so column
# names match the reference notebook.
data = data.rename(columns={data.columns[0]: "Date"})
data["Date"] = pd.to_datetime(data["Date"])
data = data.set_index("Date").sort_index()

INDEX_TICKERS = {"MXWO", "MXWD", "LEGATRUU", "HFRXGL"}
data = data.rename(
    columns={c: (f"{c} Index" if c in INDEX_TICKERS else f"{c} Comdty") for c in data.columns}
)

# Monster Index = 0.50 HFRX + 0.25 MSCI World + 0.25 Global Aggregate Bond
TARGET_WEIGHTS = {
    "HFRXGL Index": 0.50,
    "MXWO Index": 0.25,
    "LEGATRUU Index": 0.25,
}
FUTURES = [
    "RX1 Comdty", "TY1 Comdty", "GC1 Comdty", "CO1 Comdty",
    "ES1 Comdty", "VG1 Comdty", "NQ1 Comdty", "LLL1 Comdty",
    "TP1 Comdty", "DU1 Comdty", "TU2 Comdty",
]
N_ASSETS = len(FUTURES)

returns_all = data.pct_change()
target_returns = sum(returns_all[c] * w for c, w in TARGET_WEIGHTS.items())
target_returns.name = "target"

X = returns_all[FUTURES].dropna()
y = target_returns.loc[X.index].dropna()
X = X.loc[y.index]

print(f"Futures returns X: {X.shape}, target returns y: {y.shape}")
print(f"Date range: {X.index.min().date()} -> {X.index.max().date()}")


## M5.1 Feature engineering

Construct $\phi_t$ from:

- trailing $\{4, 12, 52\}$-week futures returns,
- trailing $\{12, 52\}$-week realized volatility per future,
- equity / rate / target-vol regime indicators,
- the prior Elastic-Net-style Ridge weight $w^{\mathrm{EN}}_{t-1}$ as a warm start.

$\phi_t$ is shifted by one period so it never sees contemporaneous returns.


In [ ]:
@dataclass
class FeatureConfig:
    """M5.1 feature design knobs."""
    return_lookbacks: tuple = (4, 12, 52)
    vol_lookbacks: tuple = (12, 52)
    use_regime: bool = True
    use_warmstart: bool = True
    warmstart_window: int = 52
    warmstart_alpha: float = 1e-3


def _compounded_return(s: pd.Series, k: int) -> pd.Series:
    return (1.0 + s).rolling(k).apply(np.prod, raw=True) - 1.0


def _ridge_warmstart(X: pd.DataFrame, y: pd.Series, window: int, alpha: float) -> pd.DataFrame:
    """Trailing-window Ridge fit with MinMax normalization, mirroring the EN baseline."""
    cols = [f"w_en_{c}" for c in X.columns]
    out = pd.DataFrame(index=X.index, columns=cols, dtype=float)
    Xv, yv = X.values, y.values
    for i in range(window, len(X)):
        scaler = MinMaxScaler()
        X_tr = scaler.fit_transform(Xv[i - window : i])
        mdl = Ridge(alpha=alpha, fit_intercept=False)
        mdl.fit(X_tr, yv[i - window : i])
        out.iloc[i] = mdl.coef_ / scaler.scale_
    return out


def build_features(
    X: pd.DataFrame,
    y: pd.Series,
    data: pd.DataFrame,
    cfg: FeatureConfig = FeatureConfig(),
) -> pd.DataFrame:
    """Build phi_t for every rebalance date.

    phi_t uses information up to t-1 only (lagged by one step) so the network
    can never peek at the contemporaneous return it is asked to weight.
    """
    feats: dict[str, pd.Series] = {}

    # Trailing compounded returns per future across {4, 12, 52} weeks.
    for k in cfg.return_lookbacks:
        for c in X.columns:
            feats[f"ret_{k}w_{c}"] = _compounded_return(X[c], k)

    # Trailing realized (annualized) volatility per future.
    for k in cfg.vol_lookbacks:
        vol = X.rolling(k).std() * np.sqrt(52)
        for c in X.columns:
            feats[f"vol_{k}w_{c}"] = vol[c]

    # Regime indicators: equity trend, bond trend (rate proxy), target vol.
    if cfg.use_regime:
        mxwo_ret = data["MXWO Index"].pct_change().reindex(X.index)
        bond_ret = data["LEGATRUU Index"].pct_change().reindex(X.index)
        feats["regime_msci_12w"] = _compounded_return(mxwo_ret, 12)
        feats["regime_bond_12w"] = _compounded_return(bond_ret, 12)
        feats["regime_target_vol_12w"] = y.rolling(12).std() * np.sqrt(52)

    # Warm start: prior Elastic-Net-style weights (Ridge fit, same scaling pipeline).
    if cfg.use_warmstart:
        warm = _ridge_warmstart(X, y, cfg.warmstart_window, cfg.warmstart_alpha)
        for c in warm.columns:
            feats[c] = warm[c]

    phi = pd.DataFrame(feats)
    # Lag by one period so phi_t depends only on info available at the close of t-1.
    phi = phi.shift(1).dropna()
    return phi


## M5.2 MLP architecture

$\phi_t \in \mathbb{R}^{d_\phi} \to 64 \to 32 \to w_t \in \mathbb{R}^{11}$, no softmax (long/short allowed). An optional gross-exposure cap stands in for the VaR projection that will be wired in next week.


In [ ]:
class WeightMLP(nn.Module):
    """End-to-end weight generator: phi_t -> 64 -> 32 -> w_t in R^11.

    No softmax: long/short positions are allowed. Optional gross-exposure cap
    serves as a stand-in for the M3.2 / VaR feasibility projection.
    """

    def __init__(
        self,
        in_dim: int,
        n_assets: int = 11,
        hidden: tuple[int, ...] = (64, 32),
        dropout: float = 0.1,
        gross_cap: float | None = None,
    ):
        super().__init__()
        layers: list[nn.Module] = []
        prev = in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, n_assets))
        self.net = nn.Sequential(*layers)
        self.gross_cap = gross_cap

    def forward(self, phi: torch.Tensor) -> torch.Tensor:
        w = self.net(phi)
        if self.gross_cap is not None:
            ge = w.abs().sum(dim=-1, keepdim=True).clamp(min=1e-8)
            scale = torch.minimum(torch.ones_like(ge), self.gross_cap / ge)
            w = w * scale
        return w


def project_var_cap(weights: np.ndarray, recent_returns: np.ndarray,
                    confidence: float = 0.01, horizon: int = 4,
                    var_cap: float = 0.08) -> np.ndarray:
    """Post-hoc Gaussian VaR projection (matches the Elastic Net baseline)."""
    sigma = np.std(recent_returns @ weights)
    z = norm.ppf(confidence)
    var = -z * sigma * np.sqrt(horizon)
    if var > var_cap:
        return weights * (var_cap / var)
    return weights


### Skeleton smoke test

Instantiate the model and run a single forward pass to confirm shapes and the gross-exposure cap.


In [ ]:
# Smoke test: build phi_t, instantiate the MLP, run one forward pass.
cfg = FeatureConfig()
phi = build_features(X, y, data, cfg)
print(f"phi: {phi.shape} (T x d_phi)")
print(f"feature groups: {sorted({n.split('_')[0] for n in phi.columns})}")

# Align target with the feature frame (phi already uses lagged info up to t-1,
# so we want to predict y_t from phi_t).
y_aligned = y.loc[phi.index]
X_aligned = X.loc[phi.index]
print(f"X_aligned: {X_aligned.shape}, y_aligned: {y_aligned.shape}")

model = WeightMLP(in_dim=phi.shape[1], n_assets=N_ASSETS, gross_cap=2.0).to(device)
print(model)

phi_t = torch.tensor(phi.values, dtype=torch.float32, device=device)
with torch.no_grad():
    w_t = model(phi_t)

assert w_t.shape == (len(phi), N_ASSETS), w_t.shape
print(f"w_t: {tuple(w_t.shape)}")
print(f"avg gross exposure: {w_t.abs().sum(dim=-1).mean().item():.3f}")
print(f"sample weights w_T: {w_t[-1].cpu().numpy().round(4)}")
